In [2]:
import pandas as pd

db_uri = "sqlite:////Users/eidens/Projects/mecadoi-archives/batch/batch.sqlite3"

query = """
SELECT *
FROM deposition_attempt
WHERE deposition LIKE '%https://orcid.org/http://orcid.org%'
"""

df = pd.read_sql_query(query, db_uri)
df

,id_parsed_file,deposition,attempted_at,succeeded,id,verification_failed,status
0,1053,"<doi_batch xmlns=""http://www.crossref.org/sche...",2024-09-04 05:23:01.723704,None,8770,None,1
1,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-01-05 05:23:02.471156,None,10454,None,20
2,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-01-06 02:23:32.963375,None,10561,None,20
3,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-01-13 02:23:31.672141,None,10672,None,20
4,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-01-20 02:23:32.376670,None,10785,None,20
5,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-01-27 02:23:36.467056,None,10903,None,20
6,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-02-03 02:23:29.000194,None,11027,None,20
7,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-02-10 02:23:29.733790,None,11158,None,20
8,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-02-17 02:23:29.832774,None,11294,None,20
9,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-02-24 02:23:29.802292,None,11434,None,1


In [5]:
to_fix = df.iloc[9:]
to_fix

,id_parsed_file,deposition,attempted_at,succeeded,id,verification_failed,status
9,1173,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-02-24 02:23:29.802292,None,11434,None,1
10,1515,"<doi_batch xmlns=""http://www.crossref.org/sche...",2025-11-19 05:23:01.913146,None,29088,None,1


In [20]:
from difflib import unified_diff

original_depositions = to_fix['deposition'].tolist()
corrected_depositions = [
    deposition.replace('https://orcid.org/http://orcid.org', 'https://orcid.org')
    for deposition in original_depositions
]
for i, (original, fixed) in enumerate(zip(original_depositions, corrected_depositions)):

    print(f'Deposition {i}:')

    dois_affected = [line for line in original.splitlines() if "<doi>" in line.lower()]
    for line in dois_affected:
        print(line)

    diff = unified_diff(original.splitlines(), fixed.splitlines())
    print('\n'.join(diff))

    print()

Deposition 0:
        <doi>10.15252/rc.2025414611</doi>
        <doi>10.15252/rc.2025737807</doi>
        <doi>10.15252/rc.2025897098</doi>
--- 

+++ 

@@ -142,7 +142,7 @@

               <institution_department>Department of Genetics, Faculty of Biology</institution_department>
             </institution>
           </affiliations>
-          <ORCID authenticated="true">https://orcid.org/http://orcid.org/0000-0002-5512-0443</ORCID>
+          <ORCID authenticated="true">https://orcid.org/0000-0002-5512-0443</ORCID>
         </person_name>
       </contributors>
       <titles>

Deposition 1:
        <doi>10.15252/rc.2025299272</doi>
        <doi>10.15252/rc.2025081109</doi>
        <doi>10.15252/rc.2025526950</doi>
        <doi>10.15252/rc.2025235721</doi>
--- 

+++ 

@@ -153,7 +153,7 @@

               <institution_department>Cell Architecture Laboratory</institution_department>
             </institution>
           </affiliations>
-          <ORCID authenticated="true">https://orci

In [ ]:
from mecadoi.crossref.api import deposit

responses = []
for deposition in corrected_depositions:
    response = deposit(deposition, verbose=1)
    responses.append(response)

POST https://doi.crossref.org/servlet/deposit
Content-Length: 8143
Content-Type: multipart/form-data; boundary=133b87bdca84955c8de281bc8f878b1e
--133b87bdca84955c8de281bc8f878b1e
Content-Disposition: form-data; name="login_id"

***
--133b87bdca84955c8de281bc8f878b1e
Content-Disposition: form-data; name="login_passwd"

***
--133b87bdca84955c8de281bc8f878b1e
Content-Disposition: form-data; name="fname"; filename="deposition.xml"

<doi_batch xmlns="http://www.crossref.org/schema/5.3.1" xmlns:rel="http://www.crossref.org/relations.xsd" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" version="5.3.1" xsi:schemaLocation="http://www.crossref.org/schema/5.3.1 http://www.crossref.org/schemas/crossref5.3.1.xsd">
  <head>
    <doi_batch_id>rc.1740363809815865604</doi_batch_id>
    <timestamp>1740363809815865604</timestamp>
    <depositor>
      <depositor_name>EMBO</depositor_name>
      <email_address>eidens@embl.de</email_address>
    </depositor>
    <registrant>EMBO</registrant>
  </head

In [ ]:
responses

['\n\n\n\n<html>\n<head><title>SUCCESS</title>\n</head>\n<body>\n<h2>SUCCESS</h2>\n<p>Your batch submission was successfully received.</p>\n</body>\n</html>\n',
 '\n\n\n\n<html>\n<head><title>SUCCESS</title>\n</head>\n<body>\n<h2>SUCCESS</h2>\n<p>Your batch submission was successfully received.</p>\n</body>\n</html>\n']

In [ ]:
# write deposition attempt fixes back to the database?